# ShowDiffraction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/showdiffraction.ipynb)

In [13]:
import numpy as np

from quantem.widget import ShowDiffraction

rng = np.random.default_rng(0)
size = 256
center = (size - 1) / 2
rows, cols = np.mgrid[0:size, 0:size]
radius = np.hypot(rows - center, cols - center)


def bragg_lattice(rotation_deg=0.0, spacing_px=28.0):
    angle = np.deg2rad(rotation_deg)
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    pattern = np.zeros((size, size), np.float32)
    for h in range(-4, 5):
        for k in range(-4, 5):
            spot_row = center + (h * cos_a - k * sin_a) * spacing_px
            spot_col = center + (h * sin_a + k * cos_a) * spacing_px
            amplitude = 6.0 if h == 0 and k == 0 else 1.0 / (1 + 0.4 * (h * h + k * k))
            pattern += amplitude * np.exp(-((rows - spot_row) ** 2 + (cols - spot_col) ** 2) / 8.0)
    return pattern


single_crystal = bragg_lattice()
tilt_series = np.stack([bragg_lattice(angle) for angle in (0, 4, 8, 12)]).astype(np.float32)

ring_radii_px = (34.0, 55.0, 78.0, 96.0)
polycrystalline = 3.0 * np.exp(-(radius ** 2) / 32.0)
for ring_radius in ring_radii_px:
    polycrystalline += 2.0 * np.exp(-((radius - ring_radius) ** 2) / 9.68)
polycrystalline = (polycrystalline + 0.01 * rng.random((size, size))).astype(np.float32)

## Single-crystal SAED

In [14]:
saed = ShowDiffraction(
    single_crystal,
    center=(center, center),
    bf_radius=14,
    k_pixel_size=0.018,
    title="Single-crystal SAED",
    offline=True,
    verbose=False,
)
saed.detect_spots(max_spots=12)
saed

ShowDiffraction(shape=(1, 256, 256), sampling=(1.0 Å, 0.018 1/Å), frame=0/1, spots=12, title='Single-crystal SAED')

## Polycrystalline rings

In [15]:
powder = ShowDiffraction(polycrystalline, title="Polycrystalline", offline=True, verbose=False)
powder.detect_rings(max_rings=4)
powder.calibrate_from_ring(powder.rings[0]["radius_px"], d_known=2.355)
powder

ShowDiffraction(shape=(1, 256, 256), sampling=(1.0 Å, 0.006363226984273428 1/Å), frame=0/1, title='Polycrystalline')

## Tilt series

In [16]:
tilt = ShowDiffraction(
    tilt_series,
    center=(center, center),
    bf_radius=14,
    k_pixel_size=0.018,
    title="Tilt series",
    offline=True,
    verbose=False,
)
tilt

ShowDiffraction(shape=(4, 256, 256), sampling=(1.0 Å, 0.018 1/Å), frame=0/4, title='Tilt series')

## Real data: magnetite rings

In [ ]:
import pathlib

from quantem.widget import showdiffraction

magnetite_pattern = np.load(pathlib.Path(showdiffraction.__file__).parent / "data" / "fe3o4_saed_512.npy")

magnetite = ShowDiffraction(magnetite_pattern, title="Fe3O4 (magnetite)", offline=True, verbose=False)
magnetite.auto_detect_center()
magnetite.detect_rings(max_rings=5)
inner_ring = min(magnetite.rings, key=lambda ring: ring["radius_px"])
magnetite.calibrate_from_ring(inner_ring["radius_px"], d_known=2.532)
magnetite.dp_colormap = "viridis" 
magnetite

ShowDiffraction(shape=(1, 512, 512), sampling=(1.0 Å, 0.00522249102320405 1/Å), frame=0/1, title='Fe3O4 (magnetite)')

In [ ]:
# Optional
export_path = saed.export_html("showdiffraction_saed.html", title="Single-crystal SAED")
export_path.name

'showdiffraction_saed.html'